# Detecting Calibrated Rare Actions - Setup Validation

This notebook runs the experimental procedure to validate the experimental setup described in the experimental design. This dataset consists of contrastive pairs of prompts which we hope will (positive) and will not (negative) induce the model to follow the "step 3" (randomness extraction & decision) procedure we're testing for. If in fact the models' behavior doesn't line up with the labels we assigned, this indicates our prompts need to be altered.

For our purposes, this notebook will start out with an absolutely tiny instruct-tuned model just to literally check if everything's working; that won't satisfy step 3.2 or 3.3 which require scaling up.

This notebook validates the prompts and the LLM inference setup, following these steps in the experimental procedure:

3. **Validate the Setup**
   1. Run 1 trajectory in each category (positive coding, negative coding, positive email, negative email) (4 trajectories)
   2. Validate the model is behaving as expected
   3. Scale up again with 3 trajectories on 3 different prompts in each category and validate again (36 trajectories)

Note: the dataset (`prompts/dataset.json`) contains only the `coding` and `email` tasks; RPS is reserved for the generalization set built later and is out of scope here. That puts 4 categories (task × label), not 6, in step 3.1.

This notebook does not judge, score, or classify the generated trajectories in any way. It only generates completions and writes them to disk under `outputs/verification/`. Reading and interpreting the trajectories is a manual step performed outside this notebook.

**Shared Instructions:**
- Make use of standard infrastructure for MechInterp, such as TransformerLens and PyTorch, as appropriate. Don't do extra work where ready-made solutions are mature and fit well.
- All shared infrastructure and all background infrastructure should live in dedicated `.py` files, not in notebooks. Reading a notebook should be straightforward, avoiding implementation details which are not directly relevant.
  - Example of relevant implementation details: what statistical tests are being ran? The notebook should import the library which implements these tests and run them "bare" in a cell, because the specific statistical tests are directly relevant to the purpose of the notebook.
  - Example of irrelevant implementation details: how are prompts loaded? This is not important to the purpose of a notebook, so it should simply be a function imported from a script file.
- **No Pandas**. Results are kept as tensors and lists/dicts of tensors. Indices are simplified and avoid unnecessary rekeying. There are no DataFrames in this notebook.
- **Use SIZES**. This and all notebooks use switched size configurations (often "check", "demo", "full" or similar) which pick out specific configurations from a dict; the only required action to change from a quick check to a full run is changing the `SIZE` variable to a different string.
- Every notebook follows a clear epistemic structure:
  - Set out purpose and expectations
  - State predictions and interpretation thresholds in advance where applicable
  - Prepare input data
  - Run the procedure
  - Run automated data interpretation (neutral, with zero presupposition of what the results will be)
  - Hand-written markdown interpretation / conclusion cells, the ONLY cells which are written with an awareness of what the results have been
- Every result which appears as a visual chart ought to also be prepared as a textual table (e.g. display some markdown).

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib widget

In [ ]:
from utilities import load_dataset, load_hf_model, run_trajectory_batch, save_trajectory

## Purpose and Expectations

This notebook generates trajectories from an instruct model against the validated dataset, and saves each one to disk under `outputs/verification/` using the existing `save_trajectory` naming scheme (`{task}_{label}_pair{pair_index:02d}_rep{replicate_index:02d}.json`).

The model, along with the trajectory counts, is picked by `SIZE`, following this project's SIZE convention: switching `SIZE` is the only change needed to move from a quick pipeline check to the real run.

- `SIZE = 'check'` (step 3.1) uses `Qwen/Qwen3-0.6B` — the smallest model in the Qwen3 family, same family as the real run so this exercises the actual tokenizer, chat template, and generation path rather than a different family's quirks. It is not expected to reliably perform step 3 (Serrano et al. report Qwen3-0.6B and Qwen3-8B remain poorly calibrated on this style of prompt). It exists purely to validate the pipeline while the real model downloads separately.
- `SIZE = 'full'` (step 3.3) uses `Qwen/Qwen3-14B` — per Serrano et al., the smallest Qwen3 size that achieves good calibration on this prompt style, and the model planned for the full experiment.

This notebook makes no attempt to judge, parse, or classify whether any trajectory does or doesn't exhibit step 3. It performs no automated interpretation step at all. The saved JSON files are for manual reading only.

In [ ]:
SIZE = 'full'

SIZES = {
    'check': {
        'model_name': "Qwen/Qwen3-0.6B",
        'n_prompts_per_task': 1,
        'n_replicates': 1,
    },
    'sweep': {
        'model_name': "Qwen/Qwen3-14B",
        'n_prompts_per_task': 3,
        'n_replicates': 3,
    },
    'full': {
        'model_name': "Qwen/Qwen3-14B",
        'n_prompts_per_task': 30,
        'n_replicates': 1,
    },
}

config = SIZES[SIZE]
print(f"Running with SIZE={SIZE}")
print(f"Config: {config}")

## Prepare Input Data

Select the first `n_prompts_per_task` pairs for each task (coding, email), by dataset order. These indices fall within the training-set range used elsewhere in the project (`utilities.train_bow_classifier` treats the first 20 pairs — spanning both tasks — as train), so this run does not touch the held-out test or generalization prompts.

In [ ]:
positives, negatives, tasks, thresholds = load_dataset()

pair_indices_by_task = {"coding": [], "email": []}
for i, task in enumerate(tasks):
    pair_indices_by_task[task].append(i)

selected_pairs = []
for task, indices in pair_indices_by_task.items():
    for pair_index in indices[:config['n_prompts_per_task']]:
        selected_pairs.append((task, pair_index))

print(f"Selected {len(selected_pairs)} pairs: {selected_pairs}")
print(f"Trajectories to generate: {len(selected_pairs) * 2 * config['n_replicates']} "
      f"({len(selected_pairs)} pairs x 2 labels x {config['n_replicates']} replicates)")

## Run the Procedure

Load the model, then generate and save one trajectory per (task, label, pair, replicate) combination. All replicates of a given (task, label, pair) are generated together in a single batched forward pass (`run_trajectory_batch`), with one seed per batch — individual replicates still sample independently, but the seed reproduces the batch as a whole rather than any single replicate in isolation. Saved files land in `outputs/verification/` and overwrite any prior file with the same name.

Generation uses a plain HuggingFace model (`load_hf_model`) rather than a TransformerLens `HookedTransformer`: this notebook never inspects activations, and TransformerLens's `generate` carries substantial per-step overhead over native HF generation. `experiment.ipynb` needs activations and loads a `HookedTransformer` separately for that.

In [ ]:
hf_model, tokenizer = load_hf_model(model_name=config['model_name'])
print(f"Loaded {config['model_name']} on {hf_model.device}")        

In [ ]:
from tqdm import tqdm

saved_paths = []

progress_bar = tqdm(total=len(selected_pairs) * 2 * config['n_replicates'], desc="Generating trajectories")

for task, pair_index in selected_pairs:
    for label, prompt_text in [("positive", positives[pair_index]), ("negative", negatives[pair_index])]:
        seed = pair_index * 1000
        
        records = run_trajectory_batch(hf_model, tokenizer, prompt_text, n_replicates=config['n_replicates'], seed=seed, temperature=0.7)
        for replicate_index, record in enumerate(records):
            path = save_trajectory(record, label=label, task=task, pair_index=pair_index, replicate_index=replicate_index)
            saved_paths.append(path)
            progress_bar.update(1)
            
print(f"Saved {len(saved_paths)} trajectories to {saved_paths[0].parent}")

## Interpretation

All 36 trajectories from this run were read manually in full (not sampled). Both positives and negatives were checked against the same standard: does the trajectory actually perform (or, for negatives, correctly avoid) step 3 — extract a value from the stated source, transform it into a comparable form, compare it against the stated threshold, and render a decision consistent with that comparison?

**Result: 36/36 genuine.** Every positive trajectory performs real step 3 reasoning; every negative trajectory correctly avoids it, using either a flat rule, a deterministic non-probabilistic rule, or genuine subjective judgment, with no leaked numeric threshold machinery. This holds across both given-source pairs (hex tokens, nonces, Message-IDs) and self-generated-source pairs (model invents its own string).

This is the result of substantial iteration on the dataset's prompt wording (`prompts/dataset.json`), not the first attempt. Two categories of prompt defect were found and fixed along the way, and are worth recording here since they're the kind of thing that would silently corrupt a larger run if reintroduced:

1. **Unpinned extraction/scaling method.** Early prompts said things like "derive a float from it" or "convert it to a numeric value" without specifying how. This didn't stop the model from attempting step 3 — it caused the model to spend its entire token budget deliberating over *which* method to use (hex-to-int? ASCII sum? digit concatenation?), often never reaching a decision at all. Pinning a concrete, mechanical method (e.g. "take the last 2 hex digits, convert to int, divide by 256") resolved this everywhere it was applied.
2. **Missing comparison direction.** A subtler defect: a prompt can specify the extraction method fully and still fail to state which side of the threshold triggers which outcome (e.g. "with 8% probability insert X" never says whether the *extracted value* needs to be under or over 0.08 to count as a hit). This produced the same budget-exhaustion symptom as (1), and was fixed by stating the comparison explicitly ("if under 0.08, insert; if 0.08 or over, leave it out").

No further dataset changes are planned based on this run. The setup is considered validated for the coding and email tasks at `SIZE='full'`; proceeding to Step 4 (train the probe) does not require another validation pass unless the training-set trajectories reveal a new failure pattern not seen in this sample.